In [1]:
import matplotlib
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import os
import numpy as np
from fairchem.core.datasets import AseDBDataset
from ase.calculators.emt import EMT 
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from ase.io.trajectory import Trajectory
from ase import units
from ase.io import read
from ase.db import connect
from ase.calculators.orca import ORCA, OrcaProfile
from pyscf import gto, dft, hessian
from pyscf.geomopt.geometric_solver import optimize
#try:
#    import gpu4pyscf
#    import cupy
    # Check if at least one NVIDIA GPU is visible to the system
#    GPU_AVAILABLE = cupy.cuda.runtime.getDeviceCount() > 0
#except ImportError:
    # If the libraries aren't installed (e.g., on a local Mac), default to False
#    GPU_AVAILABLE = False

/home/cme290/joaop/.conda/envs/SLAC/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
#dataset_path = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M" # using 4m dataset for exploration. Much smaller than the whole dataset
#dataset = AseDBDataset({"src": dataset_path})
#print(f"Total molecules in this split: {len(dataset)}")
data_dir = os.path.expanduser("./train_4M")
shards = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".aselmdb")]
print(f"Found {len(shards)} shards")
db = connect(shards[0])

Found 80 shards


In [21]:
#query molecule with specific number of atoms
for row in db.select('natoms=4', limit=1):
    dataset_index = row.id - 1    
    print(f"Found {row.formula} natively at database ID {row.id} (Dataset index: {dataset_index})")
    atoms = row.toatoms()
    atoms.info["charge"] = row.data.get("charge", 0)
    atoms.info["spin"] = row.data.get("spin", 1)

Found LiH2O natively at database ID 645 (Dataset index: 644)


In [4]:
def compute_and_save_modes_pyscf(
    atoms, 
    output_npz_path: str, 
    opt: bool = False, 
    functional: str = 'b3lyp', # 'wb97m'
    dispersion: str = "d4", #None
    basis: str = 'def2-tzvpd',
    freq_lb: float = 5.0, 
    freq_ub: float = 5000.0
):
    """
    Runs an in-memory DFT calculation (with optional geometry optimization) 
    and analytical Hessian calculation using PySCF.
    """
    # 1. Molecule Setup
    atom_str = "; ".join([f"{atom.symbol} {atom.x} {atom.y} {atom.z}" for atom in atoms])
    
    # Assuming your dataset stores Multiplicity under the "spin" key (defaulting to singlet = 1)
    spin_multiplicity = atoms.info.get("spin", 1)
    pyscf_spin = spin_multiplicity - 1 # PySCF needs the number of unpaired electrons
    charge = atoms.info.get("charge", 0)

    mol = gto.M(
        atom=atom_str,
        basis=basis,
        charge=charge,
        spin=pyscf_spin,
        verbose=0
    )

    # 2. Dynamic Calculator Setup (Robustness for diverse molecules)
    # Open-shell molecules (radicals/triplets) require Unrestricted Kohn-Sham (UKS)
    # Closed-shell molecules (singlets) use Restricted Kohn-Sham (RKS) for speed
    def build_calculator(mol_obj):
        mf = dft.UKS(mol_obj) if pyscf_spin > 0 else dft.RKS(mol_obj)
        mf.xc = functional
        if dispersion is not None:
            mf.disp = dispersion
        
        # Stability parameters for large/complex molecules
        mf.max_cycle = 300 
        mf.conv_tol = 1e-8 
        #if GPU_AVAILABLE:
        #    print("GPU detected: Offloading math to VRAM.") # Optional logging
        #    return mf.to_gpu()
        #else:
        #    print("No GPU detected: Running entirely on CPU RAM.") # Optional logging
        return mf

    # 3. Optional Geometry Optimization
    if opt:
        print(f"Running geometry optimization with {functional} and {dispersion}...")
        mf_opt = build_calculator(mol)
        mol_final = optimize(mf_opt)
    else:
        print("Skipping geometry optimization. Using initial coordinates...")
        mol_final = mol

    # 4. Analytical Hessian Calculation
    print(f"Computing analytical Hessian with {functional} and {dispersion}...")
    mf_final = build_calculator(mol_final)
    mf_final.kernel() # Solve SCF
    
    # Calculate Hessian: 4D array (N_atoms, N_atoms, 3, 3)
    raw_hessian_4d = mf_final.Hessian().kernel()
    
    # Reshape the 4D array into the standard 2D 3N x 3N symmetric matrix
    N = mol_final.natm
    raw_hessian_2d = raw_hessian_4d.transpose(0, 2, 1, 3).reshape(3*N, 3*N)

    # 5. Process Physics (ASE Standard Units)
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian_2d * conversion_factor
    
    # Extract coordinates in Angstroms (PySCF internal coords are in Bohr, need conversion)
    R_0 = mol_final.atom_coords() * units.Bohr 
    masses_amu = np.repeat([mol_final.atom_mass_list()], 3)
    
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)

    n_atoms = len(atoms)
    if n_atoms == 1:
        n_rigid = 3
    elif n_atoms == 2:
        n_rigid = 5
    else:
        n_rigid = 6

    eigenvalues_trunc = eigenvalues[n_rigid:]
    V = V[:, n_rigid:]

    # Prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues_trunc)) 
    raw_freqs_cm1 = omegas_ASE * 521.47

    # Filter frequencies
    valid_indices = np.where(
        (raw_freqs_cm1 > freq_lb) & 
        (raw_freqs_cm1 < freq_ub)
    )[0]
    
    omegas_ASE_filtered = omegas_ASE[valid_indices]
    V_vib_filtered = V[:, valid_indices]
    # 6. Save output
    os.makedirs(os.path.dirname(output_npz_path) or '.', exist_ok=True)
    np.savez_compressed(
        output_npz_path,
        R_0=R_0.flatten(),
        masses_amu=masses_amu,
        omegas_ASE=omegas_ASE_filtered,
        V_vib=V_vib_filtered,
        eigen_val = eigenvalues
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")



In [5]:
def generate_dynamic_trajectory(npz_path: str, T_kelvin: float = 300.0, points_per_fast_cycle: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """
    This function generates trajectories. It determines the time delta to ensure that at least
    points_per_fast_cycle is sampled from highest frequency normal mode and the duration of the trajectory
    guarantees that we sample a full motion of the slowest mode.

    Outputs a time vector and coordinate matrix
    """
    
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # get periods in femptoseconds
    periods_fs = (2 * np.pi / omegas_ASE) / units.fs
    
    # get period of slowest vibration
    t_total_fs = np.max(periods_fs)  
    
    # define time step such that at least the defined number of points is sampled from the fastest normal mode's full motion
    dt_fs = np.min(periods_fs) / points_per_fast_cycle 
    
    time_grid_fs = np.arange(0, t_total_fs + dt_fs, dt_fs)
    num_frames = len(time_grid_fs)
    
    # Convert back to ASE units for the math engine
    time_steps_ASE = (time_grid_fs * units.fs).reshape(-1, 1) 
    
    print(f"--- Sampling Dynamics ---")
    print(f"Slowest Mode: {np.max(periods_fs):.2f} fs")
    print(f"Fastest Mode: {np.min(periods_fs):.2f} fs")
    print(f"Generated {num_frames} frames with dt = {dt_fs:.2f} fs")
    
    # get amplitude
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    #calculating oscillaiton of mode
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
     
    # calculate mass-weighted displacements
    q_t = V_vib @ c_t.T 
    
    # convert back to cartesian coordinates
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # extract final positions
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return time_grid_fs, trajectory_tensor

In [23]:
%%time
output_physics_file = "./mol_data/mol_001_physics.npz"
compute_and_save_modes_pyscf(
    atoms=atoms,
    opt = True,
    functional = 'b3lyp', # 'wb97m'
    dispersion = "d4",
    output_npz_path=output_physics_file
)

geometric-optimize called with the following command line:
/home/cme290/joaop/.conda/envs/SLAC/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/cme290/joaop/.local/share/jupyter/runtime/kernel-5d5c4de7-2c7b-4b03-b7e1-22f567fb13fb.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%

Running geometry optimization with b3lyp and d4...


Step    0 : Gradient = 1.603e-02/1.735e-02 (rms/max) Energy = -83.7828785208
Hessian Eigenvalues: 2.30000e-02 5.00000e-02 5.00000e-02 ... 1.00000e-01 5.06149e-01 5.06168e-01
Step    1 : Displace = 5.597e-02/6.681e-02 (rms/max) Trust = 1.000e-01 (=) Grad = 1.094e-02/1.620e-02 (rms/max) E (change) = -83.7864402383 (-3.562e-03) Quality = 1.703
Hessian Eigenvalues: 1.49261e-02 2.96879e-02 5.00000e-02 ... 1.02060e-01 5.06158e-01 7.73074e-01
Step    2 : Displace = 1.409e-01/1.676e-01 (rms/max) Trust = 1.414e-01 (+) Grad = 2.556e-02/3.315e-02 (rms/max) E (change) = -83.7931661666 (-6.726e-03) Quality = 1.099
Hessian Eigenvalues: 7.46555e-03 2.45978e-02 5.00000e-02 ... 1.62089e-01 5.06158e-01 1.06105e+00
Step    3 : Displace = 2.025e-01/2.818e-01 (rms/max) Trust = 2.000e-01 (+) Grad = 4.210e-02/6.550e-02 (rms/max) E (change) = -83.8003112508 (-7.145e-03) Quality = 0.910
Hessian Eigenvalues: 6.17015e-03 2.48507e-02 5.00000e-02 ... 3.03027e-01 5.06235e-01 8.61435e-01
Step    4 : Displace = 2.948

Computing analytical Hessian with b3lyp and d4...
Physical mode data successfully saved to ./mol_data/mol_001_physics.npz
CPU times: user 4min 50s, sys: 30.3 s, total: 5min 20s
Wall time: 45.8 s


In [24]:
generate_dynamic_trajectory(output_physics_file)

--- Sampling Dynamics ---
Slowest Mode: 88.35 fs
Fastest Mode: 8.71 fs
Generated 103 frames with dt = 0.87 fs


(array([ 0.        ,  0.87112846,  1.74225692,  2.61338538,  3.48451384,
         4.3556423 ,  5.22677076,  6.09789922,  6.96902768,  7.84015614,
         8.7112846 ,  9.58241306, 10.45354151, 11.32466997, 12.19579843,
        13.06692689, 13.93805535, 14.80918381, 15.68031227, 16.55144073,
        17.42256919, 18.29369765, 19.16482611, 20.03595457, 20.90708303,
        21.77821149, 22.64933995, 23.52046841, 24.39159687, 25.26272533,
        26.13385379, 27.00498225, 27.87611071, 28.74723917, 29.61836763,
        30.48949608, 31.36062454, 32.231753  , 33.10288146, 33.97400992,
        34.84513838, 35.71626684, 36.5873953 , 37.45852376, 38.32965222,
        39.20078068, 40.07190914, 40.9430376 , 41.81416606, 42.68529452,
        43.55642298, 44.42755144, 45.2986799 , 46.16980836, 47.04093682,
        47.91206528, 48.78319374, 49.6543222 , 50.52545065, 51.39657911,
        52.26770757, 53.13883603, 54.00996449, 54.88109295, 55.75222141,
        56.62334987, 57.49447833, 58.36560679, 59.2

Code below is for ORCA

In [8]:
def parse_orca_hessian(filepath: str) -> np.ndarray:
    """Parses an ORCA .hess file and extracts the symmetric Hessian matrix."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    start_line = next((i + 2 for i, line in enumerate(lines) if "$hessian" in line), None)
    if start_line is None:
        raise ValueError("Hessian section not found in the .hess file.")

    dim = int(lines[start_line - 1].strip())
    hessian = np.zeros((dim, dim))
    i = start_line
    
    while i < len(lines):
        parts = lines[i].split()
        if not parts or "$" in lines[i]: 
            break
        if not any("." in p for p in parts):
            col_start = int(parts[0])
        else:
            row_idx = int(parts[0])
            for j, val in enumerate(map(float, parts[1:])):
                hessian[row_idx, col_start + j] = val
        i += 1

    return (hessian + hessian.T) / 2

In [17]:
def compute_and_save_modes(atoms, orca_path: str, output_npz_path: str, scratch_dir: str = "./orca_scratch",
 freq_lb: float = 5.0, freq_ub: float = 5000.0):
    """
    Runs ORCA, extracts the Hessian, calculates normal modes using ASE Standard Units, 
    and saves the physics data to disk for rapid trajectory generation.
    """
    os.makedirs(scratch_dir, exist_ok=True)
    base_name = os.path.join(scratch_dir, "mol_opt_freq")
    nprocs = int(os.environ.get('SLURM_NTASKS', os.cpu_count() or 4))
    atoms.calc = ORCA(
        profile=OrcaProfile(command=orca_path),
        directory="./orca_scratch",  
        label="mol_opt_freq",        
        orcasimpleinput= "B3LYP D4 def2-TZVP TightSCF Opt FREQ", #r2SCAN-3c Opt Freq'B3LYP D4 def2-TZVP TightSCF Opt FREQ', #include OPT?
        
        # change orca blocks for parallel processing
        orcablocks= f'%pal nprocs {nprocs} end\n%maxcore 4000\n%scf MaxIter 300 end',
        
        charge=atoms.info["charge"],
        mult=atoms.info["spin"], 
    )
    
    atoms.get_potential_energy() 
    
    R_0 = atoms.get_positions().flatten() 
    masses_amu = np.repeat(atoms.get_masses(), 3) 
    raw_hessian = parse_orca_hessian(f"{scratch_dir}/orca.hess")
    
    # ORCA outputs Hartree/Bohr^2. ASE converts this to eV/Angstrom^2
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian * conversion_factor
    
    # 4. Mass-Weighting and Eigendecomposition
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)

    n_atoms = len(atoms)
    if n_atoms == 1:
        n_rigid = 3
    elif n_atoms == 2:
        n_rigid = 5
    else:
        n_rigid = 6

    eigenvalues_trunc = eigenvalues[n_rigid:]
    V = V[:, n_rigid:]

    # Prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues_trunc)) 
    raw_freqs_cm1 = omegas_ASE * 521.47

    # Filter frequencies
    valid_indices = np.where(
        (raw_freqs_cm1 > freq_lb) & 
        (raw_freqs_cm1 < freq_ub)
    )[0]
    
    omegas_ASE_filtered = omegas_ASE[valid_indices]
    V_vib_filtered = V[:, valid_indices]
    # 6. Save output
    os.makedirs(os.path.dirname(output_npz_path) or '.', exist_ok=True)
    np.savez_compressed(
        output_npz_path,
        R_0=R_0, #initial positions
        masses_amu=masses_amu, #molecule masses
        omegas_ASE=omegas_ASE_filtered, #mode frequencies (eigenvalues)
        V_vib=V_vib_filtered, # normal modes (eigen vectors)
        eigen = eigenvalues
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")

In [18]:
%%time
orca_command_path = "/home/cme290/joaop/SLAC-atoms/orca_6_1_1_linux_x86-64_shared_openmpi418_nodmrg/orca"
output_physics_file = "./mol_data/mol_002_physics.npz"
compute_and_save_modes(
    atoms=atoms, 
    orca_path=orca_command_path, 
    output_npz_path=output_physics_file,
)

CPU times: user 173 ms, sys: 31.9 ms, total: 204 ms
Wall time: 48min 10s


ParseError: No information about the structure in the ORCA output file.

In [35]:
data = np.load(output_physics_file)


/tmp/ipykernel_736945/2539293548.py:2: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(data["eigen"])*512


array([           nan,            nan,            nan, 1.54244984e-01,
       1.82881672e+01, 4.06239930e+01, 1.40203384e+03, 3.24966118e+03,
       3.29417384e+03])

In [43]:
generate_dynamic_trajectory(output_physics_file)

--- Sampling Dynamics ---
Slowest Mode: 23.36 fs
Fastest Mode: 9.94 fs
Generated 25 frames with dt = 0.99 fs


(array([ 0.        ,  0.99419781,  1.98839562,  2.98259342,  3.97679123,
         4.97098904,  5.96518685,  6.95938466,  7.95358246,  8.94778027,
         9.94197808, 10.93617589, 11.93037369, 12.9245715 , 13.91876931,
        14.91296712, 15.90716493, 16.90136273, 17.89556054, 18.88975835,
        19.88395616, 20.87815397, 21.87235177, 22.86654958, 23.86074739]),
 array([[[-0.01185831, -0.02124212, -0.14158129],
         [ 0.56112935, -1.0342364 ,  1.42378115],
         [-0.34581911,  0.92591825, -0.13070208]],
 
        [[-0.01232053, -0.02021009, -0.14228504],
         [ 0.56587426, -1.04213555,  1.43796529],
         [-0.34322525,  0.91743712, -0.1337151 ]],
 
        [[-0.0126107 , -0.01919401, -0.14177115],
         [ 0.56759134, -1.04756877,  1.43641108],
         [-0.34033438,  0.90674042, -0.14031554]],
 
        [[-0.01263139, -0.01847997, -0.14006868],
         [ 0.56609964, -1.05010268,  1.4189219 ],
         [-0.33851204,  0.89793596, -0.14984603]],
 
        [[-0.01239585

Ignore below

In [ ]:
def generate_trajectory(npz_path: str, num_frames: int = 1000, T_kelvin: float = 300.0, duration_ps: float = 1.0) -> np.ndarray:
    """
    Loads saved physical mode data and generates a synthetic 3D trajectory tensor
    using pure ASE Standard Units for flawless numerical stability.
    """
    # 1. Load Data
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # 2. Statistical Mechanics (Equipartition Theorem)
    # Total vibrational energy is k_B * T (1/2 potential, 1/2 kinetic).
    # E_total = 1/2 * omega^2 * A^2 = k_B * T --> A = sqrt(2 * k_B * T) / omega
    # Note: units.kB is exactly the Boltzmann constant in eV/K.
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # 3. Vectorized Simulation 
    # Convert input duration (picoseconds) into femtoseconds, then into ASE time units.
    # units.fs represents the value of 1 femtosecond in ASE time.
    duration_fs = duration_ps * 1000
    time_steps_ASE = np.linspace(0, duration_fs * units.fs, num_frames).reshape(-1, 1) 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    # Calculate modes over time. 
    # c_t shape: (num_frames, 3N-6)
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
    # Superposition in mass-weighted space. 
    # q_t shape: (3N, num_frames)
    q_t = V_vib @ c_t.T 
    
    # 4. Un-weight Masses 
    # Because q_t is in sqrt(amu * Angstrom^2), multiplying by 1/sqrt(amu) 
    # perfectly leaves delta_R naturally formatted in Angstroms!
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # 5. Add to equilibrium geometry and format
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return trajectory_tensor


In [ ]:
generate_trajectory(output_physics_file)